In [ ]:
#!/usr/bin/env python3"""Run all valid combinations on ClientRecord dataset with augmented nulls."""import itertoolsimport sysimport ossys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))import pandas as pdimport numpy as npimport kagglehubfrom core.GenericDataPipeline import GenericDataPipelinefrom core.RunData import RunPipelinefrom sklearn.metrics import roc_auc_scorefrom sklearn.model_selection import train_test_splitfrom core.seed_utils import SEED, set_all_seedspd.set_option('future.infer_string', False)set_all_seeds()# --- Config ---N_TRIALS = int(sys.argv[1]) if len(sys.argv) > 1 else 10NULL_THRESHOLD = 0.05MIN_GROUP_PCT = 0.02pipeline = GenericDataPipeline()# --- Load data ---path = kagglehub.dataset_download("shilongzhuang/telecom-customer-churn-by-maven-analytics")csv_files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])data1 = pd.read_csv(os.path.join(path, csv_files[0]), encoding='latin1')data2 = pd.read_csv(os.path.join(path, csv_files[2]), encoding='latin1')df = pd.merge(data2, data1, on='Zip Code')df['Customer Status'] = df['Customer Status'].apply(lambda x: 1 if x == 'Stayed' else 0)columns_to_drop = ['Customer ID', 'Churn Category', 'Churn Reason',                    'Total Charges', 'Total Revenue', 'Total Refunds',                    'Total Long Distance Charges', 'Zip Code', 'City', 'Latitude', 'Longitude']df = df.drop(columns=columns_to_drop)df = pipeline.preprocessing(df)label = "Customer Status"print(f"Dataset shape (before augmentation): {df.shape}")# --- Augmentation: inject nulls into Contract, Monthly Charge, Payment Method ---# These are features with 0% nulls. We'll null them in ~20% of rows that already# have at least one null in the existing null features (Internet/Phone/Offer group).rng = np.random.RandomState(SEED)# Existing null features (Internet group + Phone group + Offer)existing_null_feats = ['Offer', 'Internet Type', 'Avg Monthly Long Distance Charges']has_any_null = df[existing_null_feats].isna().any(axis=1)candidate_indices = df[has_any_null].index.tolist()n_sample = min(int(0.20 * len(df)), len(candidate_indices))sampled_indices = rng.choice(candidate_indices, size=n_sample, replace=False)# Augment features: Contract, Monthly Charge, Payment Methodaugment_features = ['Contract', 'Monthly Charge', 'Payment Method']choices = rng.choice(['contract', 'charge', 'payment', 'contract+charge', 'charge+payment', 'all'],                     size=len(sampled_indices))contract_idx = sampled_indices[np.isin(choices, ['contract', 'contract+charge', 'all'])]charge_idx = sampled_indices[np.isin(choices, ['charge', 'contract+charge', 'charge+payment', 'all'])]payment_idx = sampled_indices[np.isin(choices, ['payment', 'charge+payment', 'all'])]df.loc[contract_idx, 'Contract'] = np.nandf.loc[charge_idx, 'Monthly Charge'] = np.nandf.loc[payment_idx, 'Payment Method'] = np.nanprint("\n=== After Augmentation ===")print("Null ratios (features with nulls):")for col in df.columns:    nr = df[col].isna().mean()    if nr > 0.01:        print(f"  {col}: {nr:.2%}")print(f"\nN_TRIALS per model: {N_TRIALS}")print(f"Target distribution:\n{df[label].value_counts()}")# --- OCDS Feature Comparison ---print(f"\n{'='*120}")print("OCDS FEATURE COMPARISON")print(f"{'='*120}")from OCDS.ocds_feature_comparison import OCDSFeatureComparison# Define extended features (fill in the list with feature names)extended_features = []  # TODO: Add extended feature names hereprint(f"Extended features: {extended_features}")print(f"Running OCDS comparison...")# Initialize and run OCDS comparisonocds_comparison = OCDSFeatureComparison(    df=df.copy(),    features_extended=extended_features,    target_col=label,    n_estimators=10,        max_depth=6,    window_size=100,    drift_threshold=0.05,                )# Run the comparisonresult = ocds_comparison.run()print(f"\n{'='*120}")print("OCDS SUMMARY")print(f"{'='*120}")print(f"AUC (Basic features only):  {result['auc_without_extended']:.4f}")print(f"AUC (All features):         {result['auc_with_all']:.4f}")print(f"Difference:                 {result['difference']:+.4f}")print(f"{'='*120}")print("\nOCDS Analysis Complete!")